In [3]:
!pip install -q transformers datasets accelerate \
    sentence-transformers scikit-learn \
    evaluate scikit-learn pandas numpy tqdm

In [4]:
import os
import random
import numpy as np
import torch
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score, accuracy_score

from sentence_transformers import SentenceTransformer

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

GPU: Tesla T4


In [5]:
dataset = load_dataset("coastalchp/ledgar")

print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/3.39k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.54MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.65MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
})
{'text': 'Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors assigns, expressly waives presentment, protest, demand, notice of dishonor, nonpayment, maturity, presentment purpose accelerating diligence collection.', 'label': 97}


In [6]:
print(dataset["train"].features)
print(dataset["train"].column_names)

print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

{'text': Value('string'), 'label': ClassLabel(names=['Adjustments', 'Agreements', 'Amendments', 'Anti-Corruption Laws', 'Applicable Laws', 'Approvals', 'Arbitration', 'Assignments', 'Assigns', 'Authority', 'Authorizations', 'Base Salary', 'Benefits', 'Binding Effects', 'Books', 'Brokers', 'Capitalization', 'Change In Control', 'Closings', 'Compliance With Laws', 'Confidentiality', 'Consent To Jurisdiction', 'Consents', 'Construction', 'Cooperation', 'Costs', 'Counterparts', 'Death', 'Defined Terms', 'Definitions', 'Disability', 'Disclosures', 'Duties', 'Effective Dates', 'Effectiveness', 'Employment', 'Enforceability', 'Enforcements', 'Entire Agreements', 'Erisa', 'Existence', 'Expenses', 'Fees', 'Financial Statements', 'Forfeitures', 'Further Assurances', 'General', 'Governing Laws', 'Headings', 'Indemnifications', 'Indemnity', 'Insurances', 'Integration', 'Intellectual Property', 'Interests', 'Interpretations', 'Jurisdictions', 'Liens', 'Litigations', 'Miscellaneous', 'Modifications'

In [7]:
label_feature = dataset["train"].features["label"]

label_names = label_feature.names
num_labels = len(label_names)

id2label = {
    i: name for i, name in enumerate(label_names)
}

label2id = {
    name: i for i, name in enumerate(label_names)
}

print("Number of classes:", num_labels)
print(label_names)

Number of classes: 100
['Adjustments', 'Agreements', 'Amendments', 'Anti-Corruption Laws', 'Applicable Laws', 'Approvals', 'Arbitration', 'Assignments', 'Assigns', 'Authority', 'Authorizations', 'Base Salary', 'Benefits', 'Binding Effects', 'Books', 'Brokers', 'Capitalization', 'Change In Control', 'Closings', 'Compliance With Laws', 'Confidentiality', 'Consent To Jurisdiction', 'Consents', 'Construction', 'Cooperation', 'Costs', 'Counterparts', 'Death', 'Defined Terms', 'Definitions', 'Disability', 'Disclosures', 'Duties', 'Effective Dates', 'Effectiveness', 'Employment', 'Enforceability', 'Enforcements', 'Entire Agreements', 'Erisa', 'Existence', 'Expenses', 'Fees', 'Financial Statements', 'Forfeitures', 'Further Assurances', 'General', 'Governing Laws', 'Headings', 'Indemnifications', 'Indemnity', 'Insurances', 'Integration', 'Intellectual Property', 'Interests', 'Interpretations', 'Jurisdictions', 'Liens', 'Litigations', 'Miscellaneous', 'Modifications', 'No Conflicts', 'No Default

In [8]:
from collections import Counter

train_labels = dataset["train"]["label"]

class_counts = Counter(train_labels)

sorted_classes = sorted(
    range(num_labels),
    key=lambda x: class_counts[x]
)

alpha = 0.65

num_low_classes = int(alpha * num_labels)

low_frequency_classes = set(
    sorted_classes[:num_low_classes]
)

print("Low-frequency classes:",
      len(low_frequency_classes))

print("Selected categories:")

for label_id in sorted_classes[:10]:
    print(
        id2label[label_id],
        class_counts[label_id]
    )

Low-frequency classes: 65
Selected categories:
Books 23
Assigns 31
Qualifications 47
Anti-Corruption Laws 106
Sanctions 118
Powers 125
Venues 131
Costs 168
Consent To Jurisdiction 175
Approvals 178


In [9]:
train_texts = dataset["train"]["text"]

tfidf = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(train_texts)

print("TF-IDF matrix:", tfidf_matrix.shape)

TF-IDF matrix: (60000, 100000)


In [10]:
retrieval_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available()
    else "cpu"
)

train_embeddings = retrieval_model.encode(
    train_texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", train_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Embedding shape: (60000, 384)


In [11]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(train_texts)

retrieval_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

retrieval_model.max_seq_length = 256

train_embeddings = retrieval_model.encode(
    train_texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [12]:
def retrieve_clause(query, query_index=None):
    query_vector = tfidf.transform([query])

    scores = (tfidf_matrix @ query_vector.T).toarray().ravel()

    if query_index is not None:
        scores[query_index] = -1

    candidate_indices = np.argpartition(
        scores, -20
    )[-20:]

    candidate_indices = candidate_indices[
        np.argsort(scores[candidate_indices])[::-1]
    ]

    query_embedding = retrieval_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    semantic_scores = (
        train_embeddings[candidate_indices]
        @ query_embedding
    )

    top5_indices = np.argsort(
        semantic_scores
    )[-5:][::-1]

    best_index = candidate_indices[top5_indices[0]]

    return train_texts[best_index]

In [13]:
tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base"
)

def truncate_retrieved_text(text, max_tokens=64):
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=max_tokens,
        add_special_tokens=False
    )

    return tokenizer.decode(
        tokens["input_ids"],
        skip_special_tokens=True
    )


def augment_text(text, retrieved_text):
    retrieved_text = truncate_retrieved_text(
        retrieved_text
    )

    return (
        "Original clause: "
        + text
        + " </s> "
        + "Related clause for reference: "
        + retrieved_text
    )

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [14]:
def process_split(split_name):
    split = dataset[split_name]

    augmented_texts = []

    for i, example in enumerate(split):
        text = example["text"]
        label = example["label"]

        if label in low_frequency_classes:

            # Exclude self-retrieval for training samples.
            query_index = (
                i if split_name == "train" else None
            )

            retrieved = retrieve_clause(
                text,
                query_index=query_index
            )

            text = augment_text(text, retrieved)

        augmented_texts.append(text)

    return augmented_texts


In [15]:
train_augmented = process_split("train")

In [16]:
val_augmented = process_split("validation")

In [17]:
test_augmented = process_split("test")

In [18]:
from datasets import Dataset

def make_tokenized_dataset(texts, labels):
    ds = Dataset.from_dict({
        "text": texts,
        "labels": labels
    })

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=512,
            padding="max_length"
        )

    ds = ds.map(
        tokenize,
        batched=True,
        remove_columns=["text"]
    )

    return ds


train_dataset = make_tokenized_dataset(
    train_augmented,
    dataset["train"]["label"]
)

val_dataset = make_tokenized_dataset(
    val_augmented,
    dataset["validation"]["label"]
)

test_dataset = make_tokenized_dataset(
    test_augmented,
    dataset["test"]["label"]
)

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to("cuda" if torch.cuda.is_available() else "cpu")

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [20]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    micro_f1 = f1_score(
        labels,
        predictions,
        average="micro"
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    }

In [21]:
training_args = TrainingArguments(
    output_dir="/content/ledgar_sra",

    num_train_epochs=10,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    learning_rate=3e-5,
    weight_decay=0.0,

    lr_scheduler_type="linear",
    warmup_steps=0,

    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,

    logging_steps=100,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=SEED
)

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3
        )
    ]
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Accuracy
1,0.489903,0.446725,0.888600,0.800016,0.888600
2,0.343339,0.362150,0.906300,0.831648,0.906300
3,0.245958,0.349287,0.912300,0.848854,0.912300
4,0.205927,0.344102,0.916400,0.858361,0.916400
5,0.158401,0.341761,0.922300,0.862514,0.922300
6,0.114293,0.383722,0.917600,0.853011,0.917600
7,0.083233,0.375277,0.926100,0.862840,0.926100
8,0.055756,0.395769,0.926900,0.878898,0.926900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Accuracy
1,0.489903,0.446725,0.888600,0.800016,0.888600
2,0.343339,0.362150,0.906300,0.831648,0.906300
3,0.245958,0.349287,0.912300,0.848854,0.912300
4,0.205927,0.344102,0.916400,0.858361,0.916400
5,0.158401,0.341761,0.922300,0.862514,0.922300
6,0.114293,0.383722,0.917600,0.853011,0.917600
7,0.083233,0.375277,0.926100,0.862840,0.926100
8,0.055756,0.395769,0.926900,0.878898,0.926900
9,0.043089,0.413799,0.925300,0.875936,0.925300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
results = trainer.evaluate(
    test_dataset
)

print("Test Results:")

for key, value in results.items():
    print(f"{key}: {value}")

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Accuracy
1,0.489903,0.446725,0.888600,0.800016,0.888600
2,0.343339,0.362150,0.906300,0.831648,0.906300
3,0.245958,0.349287,0.912300,0.848854,0.912300
4,0.205927,0.344102,0.916400,0.858361,0.916400
5,0.158401,0.341761,0.922300,0.862514,0.922300
6,0.114293,0.383722,0.917600,0.853011,0.917600
7,0.083233,0.375277,0.926100,0.862840,0.926100
8,0.055756,0.395769,0.926900,0.878898,0.926900
9,0.043089,0.420138,0.928300,0.881908,0.928300


Test Results:
eval_loss: 0.42013826966285706
eval_micro_f1: 0.9283
eval_macro_f1: 0.8819076378529396
eval_accuracy: 0.9283


In [26]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=label_names,
        zero_division=0
    )
)

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Accuracy
1,0.489903,0.446725,0.888600,0.800016,0.888600
2,0.343339,0.362150,0.906300,0.831648,0.906300
3,0.245958,0.349287,0.912300,0.848854,0.912300
4,0.205927,0.344102,0.916400,0.858361,0.916400
5,0.158401,0.341761,0.922300,0.862514,0.922300
6,0.114293,0.383722,0.917600,0.853011,0.917600
7,0.083233,0.375277,0.926100,0.862840,0.926100
8,0.055756,0.395769,0.926900,0.878898,0.926900
9,0.043089,0.420138,0.928300,0.881908,0.928300


                              precision    recall  f1-score   support

                 Adjustments       0.94      0.95      0.95        88
                  Agreements       0.79      0.54      0.64        48
                  Amendments       0.95      0.94      0.94       224
        Anti-Corruption Laws       0.92      1.00      0.96        23
             Applicable Laws       0.73      0.75      0.74        53
                   Approvals       0.60      0.81      0.69        26
                 Arbitration       0.90      0.98      0.94        47
                 Assignments       0.96      0.94      0.95       195
                     Assigns       0.25      0.25      0.25         4
                   Authority       0.89      0.81      0.85        62
              Authorizations       0.99      0.89      0.94        90
                 Base Salary       1.00      0.98      0.99       112
                    Benefits       0.92      0.81      0.86        81
             Bindin

In [27]:
save_path = "/content/ledgar_sra_final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved: /content/ledgar_sra_final


In [28]:
trainer.save_model(
    "/content/drive/MyDrive/LEDGAR_SRA"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/LEDGAR_SRA"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/LEDGAR_SRA/tokenizer_config.json',
 '/content/drive/MyDrive/LEDGAR_SRA/tokenizer.json')